In [ ]:
import time, copy
import numpy as np
import torch
import torch.nn as nn
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, Subset
from skimage.metrics import structural_similarity as ssim_fn

SMOKE_TEST = False
N_CLIENTS = 10
ROUNDS = 20
LOCAL_EPOCHS = 2
BATCH_SIZE = 128
LR = 0.05
MOMENTUM = 0.9
SAMPLES_PER_CLIENT = 2000

FIXED_CLIP_C = 1.0
BOOTSTRAP_C = 0.5
NOISE_BOUND = 1.0
DELTA = 1e-5

# eps values here are per-release Gaussian calibrations, not an accounted
# total privacy budget across rounds
EPS_LIST = [600, 200, 60]
EPS_UTILITY = [600, 200]

ATTACK_ROUND = 1
LATE_ROUND = ROUNDS
ATTACK_ITERS = 200
VICTIM_IDX = 25
SEED = 0

if SMOKE_TEST:
    ROUNDS = 2
    LATE_ROUND = 2
    EPS_UTILITY = [200]

torch.manual_seed(SEED)
np.random.seed(SEED)
DEV = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"device: {DEV}")

N_CLASS = 10
MEAN, STD = (0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616)
_M = torch.tensor(MEAN).view(1, 3, 1, 1)
_S = torch.tensor(STD).view(1, 3, 1, 1)


def unnormalise(t):
    return (t * _S.to(t.device) + _M.to(t.device)).clamp(0, 1)


class CNN(nn.Module):
    def __init__(self):
        super().__init__()
        act = nn.SiLU
        self.body = nn.Sequential(
            nn.Conv2d(3, 32, 5, padding=2, stride=2), act(),
            nn.Conv2d(32, 64, 5, padding=2, stride=2), act(),
            nn.Conv2d(64, 64, 5, padding=2, stride=1), act(),
        )
        self.fc = nn.Linear(64 * 8 * 8, N_CLASS)

    def forward(self, x):
        o = self.body(x)
        return self.fc(o.view(o.size(0), -1))


def weights_init(m):
    if isinstance(m, (nn.Conv2d, nn.Linear)):
        nn.init.kaiming_normal_(m.weight, nonlinearity="relu")
        if m.bias is not None:
            nn.init.zeros_(m.bias)


def load_data():
    train_tf = transforms.Compose([
        transforms.RandomCrop(32, padding=4),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(), transforms.Normalize(MEAN, STD)])
    eval_tf = transforms.Compose([
        transforms.ToTensor(), transforms.Normalize(MEAN, STD)])

    train = datasets.CIFAR10("./data", train=True, download=True, transform=train_tf)
    victim = datasets.CIFAR10("./data", train=True, download=True, transform=eval_tf)
    test = datasets.CIFAR10("./data", train=False, download=True, transform=eval_tf)

    idx = np.random.permutation(len(train))
    clients = [Subset(train, idx[c*SAMPLES_PER_CLIENT:(c+1)*SAMPLES_PER_CLIENT].tolist())
               for c in range(N_CLIENTS)]
    return clients, DataLoader(Subset(test, list(range(2000))), batch_size=256), victim


def gaussian_sigma(sens, eps, delta=DELTA):
    return sens * np.sqrt(2.0 * np.log(1.25 / delta)) / eps


def dp_fixed(tensors, eps, C=FIXED_CLIP_C):
    # single global clipping threshold, plain Gaussian noise
    total = torch.sqrt(sum((t ** 2).sum() for t in tensors))
    scale = min(1.0, C / (total.item() + 1e-12))
    sigma = gaussian_sigma(C, eps)
    return [t * scale + torch.randn_like(t) * sigma for t in tensors]


def dp_adaptive(tensors, history, eps, b=NOISE_BOUND):
    # per-layer clipping threshold from a sliding window of past norms,
    # noise truncated to [-b, b]. threshold uses only past norms so the
    # current update doesn't leak into its own clip.
    out = []
    for i, t in enumerate(tensors):
        C_layer = float(np.mean(history[i])) if history[i] else BOOTSTRAP_C
        C_layer = max(C_layer, 1e-8)
        n = t.norm().item()
        scale = min(1.0, C_layer / (n + 1e-12))
        noise = torch.clamp(torch.randn_like(t) * gaussian_sigma(C_layer, eps), -b, b)
        out.append(t * scale + noise)
        history[i].append(n)
        history[i][:] = history[i][-3:]
    return out


def local_train(global_model, dataset, dp_mode, eps, history, dp_timer):
    model = copy.deepcopy(global_model).to(DEV)
    opt = torch.optim.SGD(model.parameters(), lr=LR, momentum=MOMENTUM)
    crit = nn.CrossEntropyLoss()
    loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)
    start = [p.detach().clone() for p in global_model.parameters()]

    model.train()
    for _ in range(LOCAL_EPOCHS):
        for xb, yb in loader:
            xb, yb = xb.to(DEV), yb.to(DEV)
            opt.zero_grad()
            crit(model(xb), yb).backward()
            opt.step()

    delta = [p.detach() - s for p, s in zip(model.parameters(), start)]
    if dp_mode == "none":
        return delta

    if DEV.type == "cuda":
        torch.cuda.synchronize()
    t0 = time.perf_counter()
    delta = dp_fixed(delta, eps) if dp_mode == "fixed" else dp_adaptive(delta, history, eps)
    if DEV.type == "cuda":
        torch.cuda.synchronize()
    dp_timer.append(time.perf_counter() - t0)
    return delta


def evaluate(model, loader):
    model.eval()
    correct = total = 0
    with torch.no_grad():
        for xb, yb in loader:
            xb, yb = xb.to(DEV), yb.to(DEV)
            correct += (model(xb).argmax(1) == yb).sum().item()
            total += yb.size(0)
    return correct / total


def run_fl(clients, test_loader, dp_mode, eps, init_state, snapshot_rounds=()):
    model = CNN().to(DEV)
    model.load_state_dict(copy.deepcopy(init_state))
    histories = [[[] for _ in model.parameters()] for _ in range(N_CLIENTS)]
    dp_timer, round_timer, acc_curve, snaps = [], [], [], {}

    for r in range(1, ROUNDS + 1):
        if DEV.type == "cuda":
            torch.cuda.synchronize()
        r0 = time.perf_counter()
        deltas = [local_train(model, clients[c], dp_mode, eps, histories[c], dp_timer)
                  for c in range(N_CLIENTS)]
        with torch.no_grad():
            for i, p in enumerate(model.parameters()):
                p += torch.stack([d[i] for d in deltas]).mean(0)
        if DEV.type == "cuda":
            torch.cuda.synchronize()
        round_timer.append(time.perf_counter() - r0)

        acc = evaluate(model, test_loader)
        acc_curve.append(acc)
        if r in snapshot_rounds:
            snaps[r] = copy.deepcopy(model)
        if r % 5 == 0 or r <= 2:
            print(f"  [{dp_mode} eps={eps}] round {r:2d}  acc {acc:.4f}", flush=True)
    return model, acc_curve, snaps, dp_timer, round_timer


def dlg_attack(model, target, shape, iters=ATTACK_ITERS, label=""):
    torch.manual_seed(SEED)
    dx = torch.randn(shape, device=DEV).requires_grad_(True)
    dy = torch.randn((1, N_CLASS), device=DEV).requires_grad_(True)
    opt = torch.optim.LBFGS([dx, dy], lr=1.0)
    target = [t.detach() for t in target]

    for it in range(iters):
        def closure():
            opt.zero_grad()
            loss = torch.mean(torch.sum(
                -torch.softmax(dy, -1) * torch.log_softmax(model(dx), -1), 1))
            dg = torch.autograd.grad(loss, model.parameters(), create_graph=True)
            diff = sum(((a - b) ** 2).sum() for a, b in zip(dg, target))
            diff.backward()
            return diff
        try:
            opt.step(closure)
        except Exception as e:
            print(f"    [{label}] stopped at {it}: {e}")
            break
        if it % 100 == 0:
            print(f"    [{label}] iter {it}", flush=True)
    return dx.detach()


def victim_gradient(model, x, y):
    loss = nn.CrossEntropyLoss()(model(x), y)
    return [g.detach().clone() for g in torch.autograd.grad(loss, model.parameters())]


def to_img(t):
    return np.clip(unnormalise(t)[0].detach().cpu().numpy().transpose(1, 2, 0), 0, 1)


def score(o_t, r_t):
    o, r = to_img(o_t), to_img(r_t)
    return (float(np.mean((o - r) ** 2)),
            float(ssim_fn(o, r, channel_axis=-1, data_range=1.0)))


def main():
    clients, test_loader, victim_src = load_data()
    vx = victim_src[VICTIM_IDX][0].unsqueeze(0).to(DEV)
    vy = torch.tensor([victim_src[VICTIM_IDX][1]], device=DEV)

    torch.manual_seed(SEED)
    base = CNN().to(DEV)
    base.apply(weights_init)
    init_state = copy.deepcopy(base.state_dict())

    d = sum(p.numel() for p in base.parameters())
    print("\n" + "-" * 60)
    print(f"model params d={d}  sqrt(d)={d**0.5:.1f}  clients K={N_CLIENTS}")
    print("noise/signal ~ 4.85*sqrt(d)/(eps*sqrt(K))")
    print("-" * 60)
    for e in sorted(set(EPS_LIST + list(EPS_UTILITY)), reverse=True):
        ratio = 4.85 * (d ** 0.5) / (e * (N_CLIENTS ** 0.5))
        v = "weak" if ratio < 0.3 else "good" if ratio < 8 else "saturated"
        print(f"  eps={e:<6} sigma={gaussian_sigma(FIXED_CLIP_C,e):8.4f}"
              f"  ratio={ratio:7.2f}  {v}")
    print("-" * 60 + "\nwant ratios spanning ~0.3 to ~8\n")

    if SMOKE_TEST:
        print("=== smoke test ===")
        _, acc, snaps, _, _ = run_fl(clients, test_loader, "none", None,
                                     init_state, snapshot_rounds=(1,))
        print(f"FedAvg acc: {acc}")
        m = snaps[1]
        rec = dlg_attack(m, victim_gradient(m, vx, vy), vx.shape, label="smoke")
        mse, s = score(vx, rec)
        print(f"reconstruction @ round 1: MSE {mse:.4f}  SSIM {s:.4f}")
        fig, ax = plt.subplots(1, 2, figsize=(5, 3))
        ax[0].imshow(to_img(vx)); ax[0].set_title("Original"); ax[0].axis("off")
        ax[1].imshow(to_img(rec)); ax[1].set_title(f"SSIM {s:.3f}"); ax[1].axis("off")
        plt.tight_layout(); plt.savefig("smoke_test.png", dpi=150); plt.show()
        return

    print("=== Arm A: FedAvg (no DP) ===")
    _, accA, snapsA, _, rtA = run_fl(clients, test_loader, "none", None, init_state,
                                     snapshot_rounds=(ATTACK_ROUND, LATE_ROUND))

    util = {}
    for e in EPS_UTILITY:
        print(f"\n=== Fixed clipping, eps={e} ===")
        _, acc, _, dt, rt = run_fl(clients, test_loader, "fixed", e, init_state)
        util[("fixed", e)] = (acc, dt, rt)
        print(f"\n=== Adaptive clipping, eps={e} ===")
        _, acc, _, dt, rt = run_fl(clients, test_loader, "adaptive", e, init_state)
        util[("adaptive", e)] = (acc, dt, rt)

    print(f"\n=== Experiment 1: attack at round {ATTACK_ROUND} ===")
    mA = snapsA[ATTACK_ROUND]
    g = victim_gradient(mA, vx, vy)
    print(f"  gradient L2 norm = {torch.sqrt(sum((t**2).sum() for t in g)).item():.4f}")

    def fresh_hist():
        return [[BOOTSTRAP_C] for _ in g]

    runs = [("No DP", g)]
    runs += [(f"Fixed eps={e}", dp_fixed(g, e)) for e in EPS_LIST]
    runs += [(f"Adapt eps={e}", dp_adaptive(g, fresh_hist(), e)) for e in EPS_LIST]

    results, images = [], []
    for nm, gg in runs:
        print(f"  -> {nm}")
        rec = dlg_attack(mA, gg, vx.shape, label=nm)
        mse, s = score(vx, rec)
        results.append((nm, mse, s))
        images.append((nm, rec, s))
        print(f"     MSE {mse:.4f}  SSIM {s:.4f}")

    print(f"\n=== Experiment 2: same attack at round {LATE_ROUND} ===")
    mL = snapsA[LATE_ROUND]
    recL = dlg_attack(mL, victim_gradient(mL, vx, vy), vx.shape, label="late")
    mseL, sL = score(vx, recL)
    print(f"  round {LATE_ROUND}: MSE {mseL:.4f}  SSIM {sL:.4f}")

    print("\n=== Experiment 3: accumulated multi-epoch update (plain SGD, no DP) ===")
    tmp = copy.deepcopy(mA).to(DEV)
    opt = torch.optim.SGD(tmp.parameters(), lr=LR)
    crit = nn.CrossEntropyLoss()
    loader = DataLoader(clients[0], batch_size=BATCH_SIZE, shuffle=True)
    start = [p.detach().clone() for p in mA.parameters()]
    steps = 0
    for _ in range(LOCAL_EPOCHS):
        for xb, yb in loader:
            xb, yb = xb.to(DEV), yb.to(DEV)
            opt.zero_grad(); crit(tmp(xb), yb).backward(); opt.step()
            steps += 1
    delta = [p.detach() - s for p, s in zip(tmp.parameters(), start)]
    g_acc = [-dd / (LR * steps) for dd in delta]
    rec_a = dlg_attack(mA, g_acc, vx.shape, label="accum")
    mse_a, s_a = score(vx, rec_a)
    print(f"  accumulated ({steps} steps): MSE {mse_a:.4f}  SSIM {s_a:.4f}")

    L = ["=" * 64,
         f"CIFAR-10 | clients={N_CLIENTS} rounds={ROUNDS} "
         f"local_epochs={LOCAL_EPOCHS} batch={BATCH_SIZE}",
         f"SiLU CNN | SGD(lr={LR}, mom={MOMENTUM}) | normalised + augmented",
         "",
         "eps values are per-release Gaussian calibrations, no composition",
         "accounting across rounds. Attack is DLG with the dummy label",
         "optimised rather than analytically inferred. Adaptive arm derives",
         "its clip threshold from unprivatised gradient norms.",
         "=" * 64,
         f"\nExperiment 1 -- reconstruction at round {ATTACK_ROUND}",
         f"{'Setting':<20}{'MSE':>12}{'SSIM':>12}", "-" * 44]
    L += [f"{n:<20}{m:>12.4f}{s:>12.4f}" for n, m, s in results]
    L += [f"\nExperiment 2 -- same attack at round {LATE_ROUND}",
          f"{'Round ' + str(ATTACK_ROUND):<20}{results[0][1]:>12.4f}{results[0][2]:>12.4f}",
          f"{'Round ' + str(LATE_ROUND):<20}{mseL:>12.4f}{sL:>12.4f}",
          "\nExperiment 3 -- accumulated update, plain SGD",
          f"{'Accumulated':<20}{mse_a:>12.4f}{s_a:>12.4f}",
          "\nExperiment 4 -- final test accuracy",
          f"{'No DP':<26}{accA[-1]:>10.4f}"]
    for k in sorted(util, key=lambda x: (-x[1], x[0])):
        L.append(f"{k[0] + ' eps=' + str(k[1]):<26}{util[k][0][-1]:>10.4f}")
    L += ["\nExperiment 5 -- overhead", f"{'':<26}{'DP call':>12}{'full round':>14}",
          f"{'No DP':<26}{'-':>12}{np.mean(rtA):>13.3f}s"]
    for k in sorted(util, key=lambda x: (-x[1], x[0])):
        _, dt, rt = util[k]
        L.append(f"{k[0] + ' eps=' + str(k[1]):<26}"
                 f"{np.mean(dt)*1e6:>10.1f}us{np.mean(rt):>13.3f}s")
    L.append("=" * 64)
    out = "\n".join(L)
    print("\n" + out)
    open("results_table.txt", "w").write(out)

    n = len(images) + 1
    fig, ax = plt.subplots(1, n, figsize=(2.2 * n, 3.0))
    ax[0].imshow(to_img(vx)); ax[0].set_title("Original", fontsize=9); ax[0].axis("off")
    for i, (nm, r, s) in enumerate(images):
        ax[i+1].imshow(to_img(r))
        ax[i+1].set_title(f"{nm}\nSSIM {s:.3f}", fontsize=8); ax[i+1].axis("off")
    plt.tight_layout(); plt.savefig("fig_reconstructions.png", dpi=150, bbox_inches="tight")
    plt.show()

    plt.figure(figsize=(7, 4.5))
    rr = range(1, ROUNDS + 1)
    plt.plot(rr, accA, "k-", lw=2, label="No DP")
    for k in sorted(util, key=lambda x: (-x[1], x[0])):
        plt.plot(rr, util[k][0], "-", label=f"{k[0]} eps={k[1]}")
    plt.xlabel("Communication round"); plt.ylabel("Test accuracy")
    plt.legend(fontsize=8); plt.grid(alpha=0.3)
    plt.title(f"Privacy-utility trade-off (CIFAR-10, {N_CLIENTS} clients)")
    plt.tight_layout(); plt.savefig("fig_accuracy.png", dpi=150); plt.show()
    print("\nSaved figures and results_table.txt")


if __name__ == "__main__":
    main()
